<a href="https://colab.research.google.com/github/xmayksx1/DAX-BI/blob/main/Modelo_Semanal_Arboles_de_Decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile dashboard_demanda.py
import sys
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import xgboost as xgb
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
import pmdarima as pm
from sklearn.linear_model import LinearRegression
import itertools

from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                               QHBoxLayout, QLabel, QComboBox, QPushButton,
                               QLineEdit, QCheckBox, QListWidget, QListWidgetItem,
                               QMessageBox, QFrame, QTableWidget, QTableWidgetItem,
                               QHeaderView, QSplitter, QDateEdit, QSpinBox, QDoubleSpinBox,
                               QScrollArea, QTabWidget, QAbstractItemView, QFileDialog)
from PySide6.QtCore import Qt, Signal, QDate
from PySide6.QtGui import QColor, QBrush
from PySide6.QtWebEngineWidgets import QWebEngineView

# ==========================================
# CONFIGURACIÓN DE RUTAS
# ==========================================
RUTA_ENTRENAMIENTO = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Modelos de Prediccion\Entrenamiento de Modelo"
if not os.path.exists(RUTA_ENTRENAMIENTO):
    try: os.makedirs(RUTA_ENTRENAMIENTO)
    except: pass

FILE_LOG_ATIPICOS = os.path.join(RUTA_ENTRENAMIENTO, "log_eventos_atipicos.csv")
FILE_LOG_PROMOS = os.path.join(RUTA_ENTRENAMIENTO, "log_promociones_futuras.csv")

# ==========================================
# ARQUITECTURA PyTorch
# ==========================================
class NeuralForecaster(nn.Module):
    def __init__(self, input_size):
        super(NeuralForecaster, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

# ==========================================
# LIMPIEZA DE ATÍPICOS (PRE-PROCESAMIENTO)
# ==========================================
def aplicar_ajustes_atipicos(df_original, sku):
    if not os.path.exists(FILE_LOG_ATIPICOS): return df_original.copy()
    try:
        df_log = pd.read_csv(FILE_LOG_ATIPICOS, sep=';')
        df_log = df_log.loc[:, ~df_log.columns.str.contains('^Unnamed')]
        df_sku_ajustes = df_log[df_log['SKU'].astype(str) == str(sku)]
        df_clean = df_original.copy()
        for idx, row in df_sku_ajustes.iterrows():
            fasemana_ajuste = f"{row['Año']}-{int(row['Semana']):02d}"
            if fasemana_ajuste in df_clean['FASEMANA'].values:
                idx_original = df_clean[df_clean['FASEMANA'] == fasemana_ajuste].index[0]
                start_idx = max(0, idx_original - 4)
                if idx_original > 0:
                    promedio_previo = df_clean.iloc[start_idx:idx_original]['Valor_Real'].mean()
                    df_clean.at[idx_original, 'Valor_Real'] = promedio_previo
        return df_clean
    except Exception as e: return df_original.copy()

# ==========================================
# INYECCIÓN DE PROMOCIONES (POST-PROCESAMIENTO)
# ==========================================
def aplicar_promociones_futuras_array(prediccion_futura, fechas_futuras_dt, sku):
    if not os.path.exists(FILE_LOG_PROMOS): return prediccion_futura
    try:
        df_p = pd.read_csv(FILE_LOG_PROMOS, sep=';')
        df_p = df_p.loc[:, ~df_p.columns.str.contains('^Unnamed')]
        df_sku = df_p[df_p['SKU'].astype(str) == str(sku)]
        if df_sku.empty: return prediccion_futura

        pred_adj = np.copy(prediccion_futura)
        for i, dt in enumerate(fechas_futuras_dt):
            anio = dt.isocalendar().year
            sem = dt.isocalendar().week
            match = df_sku[(df_sku['Año'].astype(int) == anio) & (df_sku['Semana'].astype(int) == sem)]

            for _, row in match.iterrows():
                tipo = str(row['Tipo Impacto'])
                val = float(row['Valor Estimado'])
                if "Porcentaje" in tipo:
                    pred_adj[i] = pred_adj[i] * (1 + (val / 100.0))
                elif "Volumen" in tipo:
                    pred_adj[i] = pred_adj[i] + val
        return np.clip(pred_adj, 0, None)
    except Exception as e:
        print(f"Error al inyectar promoción: {e}")
        return prediccion_futura

# ==========================================
# MOTOR DE PREDICCIÓN
# ==========================================
def calcular_prediccion_modelo(serie, fechas_dt, fechas_pred_futuro, tipo_modelo, semanas_atras, semanas_futuro):
    try: decomp = seasonal_decompose(serie + 0.01, period=min(52, len(serie)//2), extrapolate_trend='freq'); decomp_seasonal = decomp.seasonal
    except: decomp_seasonal = np.zeros(len(serie))

    t_train = np.arange(len(serie))
    exog_train = np.column_stack([np.sin(2 * np.pi * t_train / 52.14), np.cos(2 * np.pi * t_train / 52.14)])
    t_fut = np.arange(len(serie), len(serie) + semanas_futuro)
    exog_fut = np.column_stack([np.sin(2 * np.pi * t_fut / 52.14), np.cos(2 * np.pi * t_fut / 52.14)])

    if tipo_modelo == "PyTorch NN":
        y_min, y_max = serie.min(), serie.max(); rango = (y_max - y_min) if (y_max - y_min) > 0 else 1
        serie_norm = (serie - y_min) / rango
        X, y = [], []
        for i in range(len(serie_norm)-4): X.append(serie_norm[i:i+4]); y.append(serie_norm[i+4])
        if len(X) == 0: return np.full(semanas_atras + semanas_futuro, np.mean(serie))
        X_t = torch.tensor(np.array(X), dtype=torch.float32); Y_t = torch.tensor(np.array(y), dtype=torch.float32).view(-1, 1)
        modelo_rn = NeuralForecaster(input_size=4); opt = optim.Adam(modelo_rn.parameters(), lr=0.01); crit = nn.MSELoss()
        best_loss = float('inf'); patience, counter = 20, 0; best_model = modelo_rn.state_dict()
        modelo_rn.train()
        for epoch in range(500):
            opt.zero_grad(); loss = crit(modelo_rn(X_t), Y_t); loss.backward(); opt.step()
            if loss.item() < best_loss: best_loss = loss.item(); counter = 0; best_model = modelo_rn.state_dict()
            else: counter += 1
            if counter >= patience: break
        if best_model: modelo_rn.load_state_dict(best_model)
        modelo_rn.eval()
        with torch.no_grad():
            pred_norm_atras = [modelo_rn(torch.tensor([serie_norm[i-4:i].tolist()], dtype=torch.float32)).item() for i in range(len(serie_norm) - semanas_atras, len(serie_norm))]
            pred_norm_fut = []; v = serie_norm[-4:].tolist()
            for _ in range(semanas_futuro): p = modelo_rn(torch.tensor([v[-4:]], dtype=torch.float32)).item(); pred_norm_fut.append(p); v.append(p)
        return np.clip(np.array(pred_norm_atras + pred_norm_fut) * rango + y_min, 0, None)

    elif tipo_modelo == "XGBoost":
        df_t = pd.DataFrame({'t': t_train, 'm': fechas_dt.dt.month, 's': exog_train[:,0], 'c': exog_train[:,1]})
        reg = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8).fit(df_t, serie)
        df_f = pd.DataFrame({'t': t_fut, 'm': fechas_pred_futuro.month, 's': exog_fut[:,0], 'c': exog_fut[:,1]})
        return np.clip(np.concatenate([reg.predict(df_t.tail(semanas_atras)), reg.predict(df_f)]), 0, None)

    elif tipo_modelo == "Prophet":
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False).fit(pd.DataFrame({'ds': fechas_dt, 'y': serie}))
        f = m.predict(m.make_future_dataframe(periods=semanas_futuro, freq='W'))
        return np.clip(np.concatenate([f['yhat'].iloc[len(serie)-semanas_atras:len(serie)].values, f['yhat'].tail(semanas_futuro).values]), 0, None)

    elif tipo_modelo == "Holt-Winters":
        tendencias = ['add', 'mul']; estacionales = ['add', 'mul']; amortiguados = [True, False]
        mejor_aic = float("inf"); mejor_mod = None
        input_serie = serie if (serie > 0).all() else serie + 0.01
        periodos_estacionales = 52 if len(serie) >= 104 else (len(serie) // 2)
        if periodos_estacionales < 2: mejor_mod = ExponentialSmoothing(input_serie, trend='add', seasonal=None).fit()
        else:
            for t, s, d in itertools.product(tendencias, estacionales, amortiguados):
                try:
                    temp_mod = ExponentialSmoothing(input_serie, trend=t, seasonal=s, seasonal_periods=periodos_estacionales, damped_trend=d).fit(optimized=True, use_brute=True)
                    if temp_mod.aic < mejor_aic: mejor_aic = temp_mod.aic; mejor_mod = temp_mod
                except: continue
            if mejor_mod is None: mejor_mod = ExponentialSmoothing(input_serie, trend='add', seasonal='add', seasonal_periods=periodos_estacionales).fit()
        return np.clip(np.concatenate([mejor_mod.fittedvalues[-semanas_atras:], mejor_mod.forecast(semanas_futuro)]), 0, None)

    elif tipo_modelo == "SARIMAX Log":
        try:
            model = SARIMAX(np.log1p(serie), exog=exog_train, order=(2, 1, 2), seasonal_order=(0, 0, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
            mod_fit = model.fit(disp=False, maxiter=200)
            return np.clip(np.expm1(np.concatenate([mod_fit.fittedvalues[-semanas_atras:], mod_fit.get_forecast(steps=semanas_futuro, exog=exog_fut).predicted_mean])), 0, np.max(serie) * 1.5)
        except: return np.full(semanas_atras + semanas_futuro, np.mean(serie))

    elif tipo_modelo == "AutoARIMA":
        try:
            mod_fit = pm.auto_arima(serie, X=exog_train, seasonal=False, stepwise=True, start_p=1, max_p=3, start_q=1, max_q=3, suppress_warnings=True)
            return np.clip(np.concatenate([mod_fit.predict_in_sample(X=exog_train)[-semanas_atras:], mod_fit.predict(n_periods=semanas_futuro, X=exog_fut)]), 0, None)
        except: return np.full(semanas_atras + semanas_futuro, np.mean(serie))

    elif tipo_modelo == "Regresión Lineal":
        X = np.arange(len(serie)).reshape(-1, 1); model = LinearRegression().fit(X, serie)
        prediccion_total = np.concatenate([model.predict(X[-semanas_atras:]), model.predict(np.arange(len(serie), len(serie) + semanas_futuro).reshape(-1, 1))])
        return np.clip(prediccion_total + np.concatenate([np.resize(decomp_seasonal, len(serie))[-semanas_atras:], np.resize(decomp_seasonal, semanas_futuro)]), 0, None)

    elif tipo_modelo == "Promedio Simple":
        prediccion_total = np.concatenate([np.full(semanas_atras, np.mean(serie)), np.full(semanas_futuro, np.mean(serie))])
        return np.clip(prediccion_total + np.concatenate([np.resize(decomp_seasonal, len(serie))[-semanas_atras:], np.resize(decomp_seasonal, semanas_futuro)]), 0, None)

    return np.full(semanas_atras + semanas_futuro, np.mean(serie))

def ejecutar_multiples_modelos_y_graficar(df_input, sku_nombre, metrica, modelos_seleccionados, semanas_futuro, semanas_eval, aplicar_promos):
    serie = df_input[metrica].values.astype(float); fechas_dt = df_input['FASEMANA_DT']; fechas_hist = fechas_dt.dt.strftime('%Y-%m-%d').values

    if len(serie) <= semanas_eval + 4:
        fig = go.Figure().update_layout(title=f"⚠️ Datos insuficientes para el SKU: {sku_nombre}.", xaxis=dict(visible=False), yaxis=dict(visible=False))
        return fig.to_html(include_plotlyjs='cdn'), [], [], [], []

    fechas_pred_futuro = pd.date_range(fechas_dt.iloc[-1] + pd.Timedelta(weeks=1), periods=semanas_futuro, freq='W-MON')
    fechas_pred_str = list(fechas_hist[-semanas_eval:]) + [f.strftime('%Y-%m-%d') for f in fechas_pred_futuro]
    fechas_eval_str = [pd.to_datetime(f).strftime('%Y-Sem%V') for f in fechas_hist[-semanas_eval:]]

    fig = go.Figure()
    htemplate = "<b>%{x|Sem %V - %b %Y}</b><br>%{data.name}: %{y:,.1f}<extra></extra>"
    fig.add_trace(go.Scatter(x=fechas_hist, y=serie, name=f'Real', line=dict(color='#1f77b4', width=2), hovertemplate=htemplate))

    colores = ['#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
    pred_dict = {}; errores_modelos = []; y_real_eval = serie[-semanas_eval:]

    for idx, mod_str in enumerate(modelos_seleccionados):
        try:
            pred = calcular_prediccion_modelo(serie, fechas_dt, fechas_pred_futuro, mod_str, semanas_eval, semanas_futuro)

            if aplicar_promos:
                pred_solo_futuro = pred[-semanas_futuro:]
                pred_futura_ajustada = aplicar_promociones_futuras_array(pred_solo_futuro, fechas_pred_futuro, sku_nombre)
                pred[-semanas_futuro:] = pred_futura_ajustada

            pred_dict[mod_str] = pred
            fig.add_trace(go.Scatter(x=fechas_pred_str, y=pred, name=mod_str, line=dict(color=colores[idx % len(colores)], dash='dot'), hovertemplate=htemplate))

            p_eval = pred[:semanas_eval]
            s_real = np.sum(y_real_eval); s_err = np.sum(np.abs(y_real_eval - p_eval))
            err_pct = (s_err / s_real) * 100 if s_real > 0 else 0.0
            errores_modelos.append({"Modelo": mod_str, "MAE": np.mean(np.abs(y_real_eval - p_eval)), "Error_Pct": err_pct, "Pred_Futuro": pred[-semanas_futuro:], "Pred_Eval": p_eval, "Total_Proyectado": np.sum(pred[-semanas_futuro:])})
        except: pass

    errores_modelos = sorted(errores_modelos, key=lambda x: x["MAE"])
    frames = []
    for i in range(1, semanas_eval + semanas_futuro + 1):
        fd = [go.Scatter(x=fechas_hist, y=serie, hovertemplate=htemplate)]
        for idx, mod_str in enumerate(modelos_seleccionados):
            if mod_str in pred_dict: fd.append(go.Scatter(x=fechas_pred_str[:i], y=pred_dict[mod_str][:i], hovertemplate=htemplate))
        frames.append(go.Frame(data=fd, name=f'f{i}'))

    fig.frames = frames
    fig.update_layout(
        title=f"📈 Proyección y Evaluación - SKU: {sku_nombre}", xaxis=dict(title="", type="date", tickformat="Sem %V<br>%b %Y"),
        yaxis_title=metrica, template="plotly_white", margin=dict(l=20, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        updatemenus=[{"type": "buttons", "direction": "right", "x": 0.0, "y": 1.15, "showactive": False,
            "buttons": [{"label": "▶ Reproducir", "method": "animate", "args": [None, {"frame": {"duration": 80, "redraw": True}, "fromcurrent": True}]},
                        {"label": "⏸ Pausar", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}]}]}]
    )
    return fig.to_html(include_plotlyjs='cdn', full_html=True), errores_modelos, [f.strftime('%Y-Sem%V') for f in fechas_pred_futuro], fechas_eval_str, y_real_eval

# ==========================================
# INTERFAZ GRÁFICA: WIDGET MULTI-SELECT
# ==========================================
class MultiSelectSearch(QWidget):
    selection_changed = Signal()
    def __init__(self, title, items):
        super().__init__()
        layout = QVBoxLayout(self)
        layout.setContentsMargins(0, 0, 0, 0)
        self.lbl = QLabel(f"<b>{title}</b>")
        self.search = QLineEdit(); self.search.setPlaceholderText("Buscar...")
        self.search.textChanged.connect(self.filter_items)
        self.chk_all = QCheckBox("Seleccionar Todos")
        self.chk_all.stateChanged.connect(self.toggle_all)
        self.list_widget = QListWidget(); self.list_widget.setFixedHeight(120)
        self.list_widget.itemChanged.connect(self.emit_signal)
        layout.addWidget(self.lbl); layout.addWidget(self.search); layout.addWidget(self.chk_all); layout.addWidget(self.list_widget)
        self._updating = False; self.load_items(items)

    def load_items(self, items):
        self._updating = True; self.list_widget.clear()
        for item in items:
            lw = QListWidgetItem(str(item)); lw.setFlags(lw.flags() | Qt.ItemIsUserCheckable); lw.setCheckState(Qt.Unchecked)
            self.list_widget.addItem(lw)
        self.chk_all.setChecked(False); self._updating = False

    def filter_items(self, text):
        terms = [t.strip().lower() for t in text.split(',') if t.strip()]
        for i in range(self.list_widget.count()):
            it = self.list_widget.item(i); it_text = it.text().lower()
            if terms: it.setHidden(not any(t in it_text for t in terms))
            else: it.setHidden(False)

    def toggle_all(self, state):
        self._updating = True
        cs = Qt.Checked if state == Qt.Checked.value else Qt.Unchecked
        for i in range(self.list_widget.count()):
            it = self.list_widget.item(i)
            if not it.isHidden(): it.setCheckState(cs)
        self._updating = False; self.emit_signal()

    def get_selected(self):
        return [self.list_widget.item(i).text() for i in range(self.list_widget.count()) if self.list_widget.item(i).checkState() == Qt.Checked]

    def emit_signal(self):
        if not self._updating: self.selection_changed.emit()

# ==========================================
# APLICACIÓN PRINCIPAL
# ==========================================
class DashboardApp(QMainWindow):
    def __init__(self, df_base):
        super().__init__()
        self.setWindowTitle("Planificación de Demanda - Aceitera El Real")
        self.resize(1500, 950); self.df_base = df_base; self.init_ui()

    def init_ui(self):
        central_widget = QWidget(); self.setCentralWidget(central_widget); main_layout = QHBoxLayout(central_widget)

        # --- PANEL IZQUIERDO INTOCABLE ---
        scroll_area = QScrollArea()
        scroll_area.setFixedWidth(380); scroll_area.setWidgetResizable(True)
        scroll_area.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        scroll_area.setStyleSheet("QScrollArea { border: none; }")

        left_panel = QWidget(); left_layout = QVBoxLayout(left_panel)
        left_layout.setContentsMargins(10, 10, 20, 10)
        scroll_area.setWidget(left_panel)

        date_layout = QHBoxLayout()
        self.date_desde = QDateEdit(); self.date_desde.setDate(QDate(2022, 1, 1)); self.date_desde.setCalendarPopup(True)
        self.date_hasta = QDateEdit(); self.date_hasta.setDate(QDate(2025, 12, 31)); self.date_hasta.setCalendarPopup(True)
        date_layout.addWidget(QLabel("Desde:")); date_layout.addWidget(self.date_desde)
        date_layout.addWidget(QLabel("Hasta:")); date_layout.addWidget(self.date_hasta)

        self.spin_eval = QSpinBox(); self.spin_eval.setRange(1, 104); self.spin_eval.setValue(26); self.spin_eval.setSuffix(" semanas")
        self.spin_futuro = QSpinBox(); self.spin_futuro.setRange(1, 104); self.spin_futuro.setValue(26); self.spin_futuro.setSuffix(" semanas")

        lista_modelos = ["PyTorch NN", "Prophet", "XGBoost", "Holt-Winters", "SARIMAX Log", "AutoARIMA", "Regresión Lineal", "Promedio Simple"]
        self.multi_modelo = MultiSelectSearch("Modelos de Predicción", lista_modelos)
        for i in range(self.multi_modelo.list_widget.count()):
            if self.multi_modelo.list_widget.item(i).text() in ["PyTorch NN", "Prophet", "XGBoost"]: self.multi_modelo.list_widget.item(i).setCheckState(Qt.Checked)

        self.drop_unidad = QComboBox(); self.drop_unidad.addItems(["Cajas", "PesoNeto", "MontoTotalDOL"])
        self.drop_bonificado = QComboBox(); self.drop_bonificado.addItems(["Incluir Bonificados (Todo)", "Excluir Bonificados"])

        self.chk_aplicar_atipicos = QCheckBox("🧽 Aplicar limpieza de Atípicos")
        self.chk_aplicar_atipicos.setChecked(True); self.chk_aplicar_atipicos.setStyleSheet("font-weight: bold; margin-top: 5px;")

        self.chk_aplicar_promos = QCheckBox("🎯 Sumar Promociones Futuras")
        self.chk_aplicar_promos.setChecked(True); self.chk_aplicar_promos.setStyleSheet("font-weight: bold; color: #1976D2;")

        proveedores = self.df_base['Proveedor'].dropna().unique().tolist()
        self.multi_prov = MultiSelectSearch("Proveedor", proveedores)
        self.multi_sublinea = MultiSelectSearch("Sublínea", [])
        self.multi_sku = MultiSelectSearch("SKU (ITMCodigo)", [])

        self.multi_prov.selection_changed.connect(self.actualizar_sublineas)
        self.multi_sublinea.selection_changed.connect(self.actualizar_skus)
        self.multi_sku.selection_changed.connect(self.sincronizar_sku_ajuste)

        self.btn_generar = QPushButton("Generar Análisis")
        self.btn_generar.setStyleSheet("background-color: #1976D2; color: white; padding: 12px; font-weight: bold; font-size: 14px;")
        self.btn_generar.clicked.connect(self.generar_grafico)

        self.btn_exportar = QPushButton("📥 Exportar a Excel")
        self.btn_exportar.setStyleSheet("background-color: #2E7D32; color: white; padding: 12px; font-weight: bold; font-size: 14px;")
        self.btn_exportar.clicked.connect(self.exportar_excel)

        left_layout.addWidget(QLabel("<b>Rango Entrenamiento</b>")); left_layout.addLayout(date_layout); left_layout.addSpacing(5)
        left_layout.addWidget(QLabel("<b>Periodo de Evaluación (Backtest)</b>")); left_layout.addWidget(self.spin_eval); left_layout.addSpacing(5)
        left_layout.addWidget(QLabel("<b>Horizonte a Futuro</b>")); left_layout.addWidget(self.spin_futuro); left_layout.addSpacing(5)
        left_layout.addWidget(QLabel("<b>Unidad de Medida</b>")); left_layout.addWidget(self.drop_unidad); left_layout.addSpacing(5)
        left_layout.addWidget(QLabel("<b>Estado del Producto</b>")); left_layout.addWidget(self.drop_bonificado); left_layout.addSpacing(5)
        left_layout.addWidget(self.chk_aplicar_atipicos); left_layout.addWidget(self.chk_aplicar_promos); left_layout.addSpacing(5)
        left_layout.addWidget(self.multi_modelo); left_layout.addWidget(self.multi_prov)
        left_layout.addWidget(self.multi_sublinea); left_layout.addWidget(self.multi_sku); left_layout.addSpacing(10)
        left_layout.addWidget(self.btn_generar)
        left_layout.addWidget(self.btn_exportar)

        # PANEL DERECHO
        right_splitter = QSplitter(Qt.Vertical)
        self.browser = QWebEngineView(); self.browser.setHtml("<h2 style='text-align: center; margin-top: 20%; color: #555;'>Seleccione parámetros.</h2>")

        self.tabs_inferiores = QTabWidget()

        # Tab 1 y 2
        tab_res = QWidget(); res_lay = QHBoxLayout(tab_res)
        self.t_error = QTableWidget(); self.t_pred = QTableWidget()
        res_lay.addWidget(self.t_error, 2); res_lay.addWidget(self.t_pred, 3)
        self.tabs_inferiores.addTab(tab_res, "🏆 Resumen y Proyección")

        tab_back = QWidget(); back_lay = QVBoxLayout(tab_back)
        self.t_backtest = QTableWidget(); back_lay.addWidget(self.t_backtest)
        self.tabs_inferiores.addTab(tab_back, "🔬 Evaluación Backtest")

        # Tab 3: Ajuste de Atípicos (RESTAURADA A VERSIÓN COMPLETA Y FUNCIONAL)
        skus_unicos = sorted([str(x) for x in self.df_base['ITMCodigo'].dropna().unique() if str(x).strip().lower() != 'nan'])
        tab_atipicos = QWidget(); atip_lay = QVBoxLayout(tab_atipicos)
        atip_form = QHBoxLayout()

        self.atip_sku = QComboBox(); self.atip_sku.setEditable(True)
        self.atip_sku.addItems(skus_unicos); self.atip_sku.completer().setFilterMode(Qt.MatchContains)

        self.atip_year = QSpinBox(); self.atip_year.setRange(2022, 2030); self.atip_year.setValue(2025)
        self.atip_week = QSpinBox(); self.atip_week.setRange(1, 53); self.atip_week.setValue(1)
        self.atip_motive = QComboBox(); self.atip_motive.addItems(["Quiebre de Stock", "Precio", "Promoción", "Feriado Nacional", "Otro"])
        self.atip_impact = QComboBox(); self.atip_impact.addItems(["A la baja", "A la alza"])

        btn_add_atip = QPushButton("Registrar")
        btn_add_atip.clicked.connect(self.registrar_atipico)
        btn_add_atip.setStyleSheet("background: #2e7d32; color: white; font-weight: bold;")

        btn_update_atip = QPushButton("Actualizar Seleccionado")
        btn_update_atip.clicked.connect(self.actualizar_atipico)
        btn_update_atip.setStyleSheet("background: #f57c00; color: white; font-weight: bold;")

        btn_del_atip = QPushButton("Eliminar")
        btn_del_atip.clicked.connect(self.eliminar_atipico)
        btn_del_atip.setStyleSheet("background: #c62828; color: white; font-weight: bold;")

        atip_form.addWidget(QLabel("SKU:")); atip_form.addWidget(self.atip_sku)
        atip_form.addWidget(QLabel("Año:")); atip_form.addWidget(self.atip_year)
        atip_form.addWidget(QLabel("Sem:")); atip_form.addWidget(self.atip_week)
        atip_form.addWidget(QLabel("Motivo:")); atip_form.addWidget(self.atip_motive)
        atip_form.addWidget(QLabel("Impacto:")); atip_form.addWidget(self.atip_impact)
        atip_form.addWidget(btn_add_atip); atip_form.addWidget(btn_update_atip); atip_form.addWidget(btn_del_atip)

        self.search_atipicos = QLineEdit()
        self.search_atipicos.setPlaceholderText("🔍 Filtrar tabla por SKU...")
        self.search_atipicos.textChanged.connect(self.filtrar_tabla_atipicos)

        self.t_atipicos = QTableWidget(); self.t_atipicos.setColumnCount(6)
        self.t_atipicos.setHorizontalHeaderLabels(["SKU", "Año", "Semana", "Motivo", "Impacto", "Fecha Registro"])
        self.t_atipicos.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.t_atipicos.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.t_atipicos.setSelectionMode(QAbstractItemView.SingleSelection)
        self.t_atipicos.setEditTriggers(QAbstractItemView.NoEditTriggers)

        self.t_atipicos.itemSelectionChanged.connect(self.cargar_datos_para_edicion)

        atip_lay.addLayout(atip_form); atip_lay.addWidget(self.search_atipicos); atip_lay.addWidget(self.t_atipicos)
        self.tabs_inferiores.addTab(tab_atipicos, "⚙️ Ajuste de Atípicos")

        # Tab 4: Promociones Futuras
        tab_promos = QWidget(); promo_lay = QVBoxLayout(tab_promos)
        promo_form = QHBoxLayout()
        self.promo_sku = QComboBox(); self.promo_sku.setEditable(True); self.promo_sku.addItems(skus_unicos); self.promo_sku.completer().setFilterMode(Qt.MatchContains)
        self.promo_year = QSpinBox(); self.promo_year.setRange(2024, 2030); self.promo_year.setValue(2026)
        self.promo_week = QSpinBox(); self.promo_week.setRange(1, 53); self.promo_week.setValue(10)
        self.promo_name = QLineEdit(); self.promo_name.setPlaceholderText("Ej. Día de la Madre")
        self.promo_type = QComboBox(); self.promo_type.addItems(["Volumen (Unidades)", "Porcentaje (%)"])
        self.promo_val = QDoubleSpinBox(); self.promo_val.setRange(-1000000, 1000000); self.promo_val.setDecimals(1)

        btn_add_promo = QPushButton("Registrar Campaña"); btn_add_promo.clicked.connect(self.registrar_promo); btn_add_promo.setStyleSheet("background: #f57c00; color: white; font-weight: bold;")
        btn_del_promo = QPushButton("Eliminar"); btn_del_promo.clicked.connect(self.eliminar_promo); btn_del_promo.setStyleSheet("background: #c62828; color: white; font-weight: bold;")
        promo_form.addWidget(QLabel("SKU:")); promo_form.addWidget(self.promo_sku); promo_form.addWidget(QLabel("Año:")); promo_form.addWidget(self.promo_year); promo_form.addWidget(QLabel("Sem:")); promo_form.addWidget(self.promo_week); promo_form.addWidget(QLabel("Campaña:")); promo_form.addWidget(self.promo_name); promo_form.addWidget(self.promo_type); promo_form.addWidget(self.promo_val); promo_form.addWidget(btn_add_promo); promo_form.addWidget(btn_del_promo)

        self.t_promos = QTableWidget(); self.t_promos.setColumnCount(6)
        self.t_promos.setHorizontalHeaderLabels(["SKU", "Año", "Semana", "Campaña", "Tipo Impacto", "Valor Estimado"])
        self.t_promos.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch); self.t_promos.setSelectionBehavior(QAbstractItemView.SelectRows)
        promo_lay.addLayout(promo_form); promo_lay.addWidget(self.t_promos)
        self.tabs_inferiores.addTab(tab_promos, "🚀 Promociones Futuras")

        right_splitter.addWidget(self.browser); right_splitter.addWidget(self.tabs_inferiores); right_splitter.setSizes([550, 300])
        main_layout.addWidget(scroll_area); main_layout.addWidget(right_splitter, stretch=1)
        self.cargar_log_atipicos()
        self.cargar_log_promos()

    def actualizar_sublineas(self):
        provs = self.multi_prov.get_selected()
        subs = sorted(self.df_base[self.df_base['Proveedor'].isin(provs)]['Sublinea'].dropna().unique().tolist()) if provs else []
        self.multi_sublinea.load_items(subs); self.actualizar_skus()

    def actualizar_skus(self):
        subs = self.multi_sublinea.get_selected(); provs = self.multi_prov.get_selected()
        df_f = self.df_base.copy()
        if provs: df_f = df_f[df_f['Proveedor'].isin(provs)]
        if subs: df_f = df_f[df_f['Sublinea'].isin(subs)]
        self.multi_sku.load_items(sorted(df_f['ITMCodigo'].unique().tolist()))

    def sincronizar_sku_ajuste(self):
        sel = self.multi_sku.get_selected()
        if sel: self.atip_sku.setCurrentText(sel[0]); self.promo_sku.setCurrentText(sel[0])

    # --- LÓGICA ATÍPICOS ---
    def filtrar_tabla_atipicos(self, text):
        t = text.lower()
        for r in range(self.t_atipicos.rowCount()):
            item = self.t_atipicos.item(r, 0)
            if item: self.t_atipicos.setRowHidden(r, t not in item.text().lower())

    def cargar_datos_para_edicion(self):
        items = self.t_atipicos.selectedItems()
        if not items: return
        row = items[0].row()
        sku = self.t_atipicos.item(row, 0).text().strip()
        anio = int(self.t_atipicos.item(row, 1).text().strip())
        semana = int(self.t_atipicos.item(row, 2).text().strip())
        motivo = self.t_atipicos.item(row, 3).text().strip()
        impacto = self.t_atipicos.item(row, 4).text().strip()

        self.atip_sku.setCurrentText(sku)
        self.atip_year.setValue(anio)
        self.atip_week.setValue(semana)
        self.atip_motive.setCurrentText(motivo)
        self.atip_impact.setCurrentText(impacto)

    def registrar_atipico(self):
        sku = self.atip_sku.currentText().strip()
        if not sku or sku.lower() == 'nan':
            QMessageBox.warning(self, "Error", "Debe especificar un SKU válido."); return
        df_new = pd.DataFrame([[sku, self.atip_year.value(), self.atip_week.value(), self.atip_motive.currentText(), self.atip_impact.currentText(), QDate.currentDate().toString(Qt.ISODate)]], columns=["SKU", "Año", "Semana", "Motivo", "Impacto", "Fecha Registro"])
        if os.path.exists(FILE_LOG_ATIPICOS):
            df_h = pd.read_csv(FILE_LOG_ATIPICOS, sep=';')
            df_h = df_h.loc[:, ~df_h.columns.str.contains('^Unnamed')]
            df_final = pd.concat([df_h, df_new]).drop_duplicates(subset=['SKU', 'Año', 'Semana'], keep='last')
        else: df_final = df_new
        df_final.to_csv(FILE_LOG_ATIPICOS, index=False, sep=';'); self.cargar_log_atipicos(); QMessageBox.information(self, "Ok", "Guardado")

    def actualizar_atipico(self):
        items = self.t_atipicos.selectedItems()
        if not items:
            QMessageBox.warning(self, "Atención", "Haga clic sobre un registro en la tabla para editarlo."); return
        row = items[0].row()
        old_sku = str(self.t_atipicos.item(row, 0).text()).strip(); old_anio = str(self.t_atipicos.item(row, 1).text()).strip(); old_sem = str(self.t_atipicos.item(row, 2).text()).strip()
        new_sku = self.atip_sku.currentText().strip(); new_anio = self.atip_year.value(); new_sem = self.atip_week.value()
        if not new_sku or new_sku.lower() == 'nan': return
        try:
            if os.path.exists(FILE_LOG_ATIPICOS):
                df = pd.read_csv(FILE_LOG_ATIPICOS, sep=';'); df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
                df['SKU'] = df['SKU'].astype(str).str.strip(); df['Año'] = df['Año'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip(); df['Semana'] = df['Semana'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
                df = df[~((df['SKU'] == old_sku) & (df['Año'] == old_anio) & (df['Semana'] == old_sem))]
                nuevo_registro = pd.DataFrame([[new_sku, new_anio, new_sem, self.atip_motive.currentText(), self.atip_impact.currentText(), QDate.currentDate().toString(Qt.ISODate)]], columns=["SKU", "Año", "Semana", "Motivo", "Impacto", "Fecha Registro"])
                df_final = pd.concat([df, nuevo_registro], ignore_index=True)
                df_final.to_csv(FILE_LOG_ATIPICOS, index=False, sep=';'); self.cargar_log_atipicos(); QMessageBox.information(self, "Éxito", "El registro ha sido actualizado.")
        except Exception as e: QMessageBox.critical(self, "Error Interno", f"Error al actualizar: {e}")

    def eliminar_atipico(self):
        items = self.t_atipicos.selectedItems()
        if not items:
            QMessageBox.warning(self, "Atención", "Por favor, seleccione una fila en la tabla."); return
        row = items[0].row()
        sku_ui = str(self.t_atipicos.item(row, 0).text()).strip(); anio_ui = str(self.t_atipicos.item(row, 1).text()).strip(); sem_ui = str(self.t_atipicos.item(row, 2).text()).strip()
        reply = QMessageBox.question(self, 'Confirmar', f"¿Eliminar el ajuste?\nSKU: {sku_ui}\nAño: {anio_ui}\nSemana: {sem_ui}", QMessageBox.Yes | QMessageBox.No, QMessageBox.No)
        if reply == QMessageBox.Yes:
            try:
                if os.path.exists(FILE_LOG_ATIPICOS):
                    df = pd.read_csv(FILE_LOG_ATIPICOS, sep=';'); df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
                    df['SKU'] = df['SKU'].astype(str).str.strip(); df['Año'] = df['Año'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip(); df['Semana'] = df['Semana'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
                    df_filtered = df[~((df['SKU'] == sku_ui) & (df['Año'] == anio_ui) & (df['Semana'] == sem_ui))]
                    df_filtered.to_csv(FILE_LOG_ATIPICOS, index=False, sep=';'); self.cargar_log_atipicos(); QMessageBox.information(self, "Éxito", "El registro fue borrado.")
            except Exception as e: QMessageBox.critical(self, "Error Interno", f"No se pudo borrar: {e}")

    def cargar_log_atipicos(self):
        if not os.path.exists(FILE_LOG_ATIPICOS): self.t_atipicos.setRowCount(0); return
        try:
            df = pd.read_csv(FILE_LOG_ATIPICOS, sep=';'); df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
            df.to_csv(FILE_LOG_ATIPICOS, index=False, sep=';'); self.t_atipicos.setRowCount(0); self.t_atipicos.setRowCount(len(df))
            cols_esperadas = ["SKU", "Año", "Semana", "Motivo", "Impacto", "Fecha Registro"]
            for col in cols_esperadas:
                if col not in df.columns: df[col] = ""
            for r, row in df.iterrows():
                for c, col_name in enumerate(cols_esperadas):
                    val = row[col_name]; val_limpio = str(val).replace('.0', '') if str(val).endswith('.0') else str(val)
                    if val_limpio.lower() == 'nan': val_limpio = ""
                    item = QTableWidgetItem(val_limpio); item.setTextAlignment(Qt.AlignCenter); self.t_atipicos.setItem(r, c, item)
        except Exception as e: print(f"Error al cargar tabla de atípicos: {e}")

    # --- LÓGICA PROMOCIONES FUTURAS ---
    def registrar_promo(self):
        sku = self.promo_sku.currentText().strip()
        if not sku or sku.lower() == 'nan': return
        df_new = pd.DataFrame([[sku, self.promo_year.value(), self.promo_week.value(), self.promo_name.text(), self.promo_type.currentText(), self.promo_val.value()]], columns=["SKU", "Año", "Semana", "Campaña", "Tipo Impacto", "Valor Estimado"])
        if os.path.exists(FILE_LOG_PROMOS):
            df_h = pd.read_csv(FILE_LOG_PROMOS, sep=';'); df_h = df_h.loc[:, ~df_h.columns.str.contains('^Unnamed')]
            df_final = pd.concat([df_h, df_new]).drop_duplicates(subset=['SKU', 'Año', 'Semana'], keep='last')
        else: df_final = df_new
        df_final.to_csv(FILE_LOG_PROMOS, index=False, sep=';'); self.cargar_log_promos(); self.promo_name.clear(); QMessageBox.information(self, "Ok", "Campaña Registrada")

    def eliminar_promo(self):
        items = self.t_promos.selectedItems()
        if not items: return
        row = items[0].row()
        sku_ui = str(self.t_promos.item(row, 0).text()).strip(); anio_ui = str(self.t_promos.item(row, 1).text()).strip(); sem_ui = str(self.t_promos.item(row, 2).text()).strip()
        if os.path.exists(FILE_LOG_PROMOS):
            df = pd.read_csv(FILE_LOG_PROMOS, sep=';'); df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
            df['SKU'] = df['SKU'].astype(str).str.strip(); df['Año'] = df['Año'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip(); df['Semana'] = df['Semana'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
            df = df[~((df['SKU'] == sku_ui) & (df['Año'] == anio_ui) & (df['Semana'] == sem_ui))]
            df.to_csv(FILE_LOG_PROMOS, index=False, sep=';'); self.cargar_log_promos()

    def cargar_log_promos(self):
        if not os.path.exists(FILE_LOG_PROMOS): self.t_promos.setRowCount(0); return
        try:
            df = pd.read_csv(FILE_LOG_PROMOS, sep=';'); df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
            self.t_promos.setRowCount(len(df))
            for r, row in df.iterrows():
                for c, col_name in enumerate(["SKU", "Año", "Semana", "Campaña", "Tipo Impacto", "Valor Estimado"]):
                    val = str(row[col_name]).replace('.0', '') if str(row[col_name]).endswith('.0') else str(row[col_name])
                    self.t_promos.setItem(r, c, QTableWidgetItem("" if val.lower() == 'nan' else val))
        except: pass

    # --- EXPORTAR A EXCEL ---
    def exportar_excel(self):
        skus = self.multi_sku.get_selected(); mods = self.multi_modelo.get_selected()
        if not skus or not mods:
            QMessageBox.warning(self, "Atención", "Seleccione al menos un SKU y un Modelo para exportar."); return

        ruta_archivo, _ = QFileDialog.getSaveFileName(self, "Guardar Exportación", "Proyeccion_Demanda.xlsx", "Excel Files (*.xlsx)")
        if not ruta_archivo: return

        self.btn_exportar.setEnabled(False); self.btn_exportar.setText("Exportando... (Esto puede tardar)"); QApplication.processEvents()

        try:
            inicio = pd.to_datetime('2024-01-01') if self.date_desde.date().year() == 2022 else pd.to_datetime(self.date_desde.date().toPython())
            fin = pd.to_datetime(self.date_hasta.date().toPython())
            metrica = self.drop_unidad.currentText(); semanas_futuro = self.spin_futuro.value(); semanas_eval = self.spin_eval.value()
            aplicar_promos = self.chk_aplicar_promos.isChecked()

            with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
                for sku in skus:
                    df_sku_base = self.df_base[self.df_base['ITMCodigo'].astype(str) == str(sku)]
                    df_sku_base = df_sku_base[(df_sku_base['FASEMANA_DT'].dt.date >= inicio.date()) & (df_sku_base['FASEMANA_DT'].dt.date <= fin.date())]

                    if self.drop_bonificado.currentText() == "Excluir Bonificados" and 'Bonificado' in df_sku_base.columns:
                        bonificados = df_sku_base['Bonificado'].astype(str).str.strip().str.lower()
                        df_sku_base = df_sku_base[bonificados.isin(['0', 'no', 'false', 'nan', '<na>']) | df_sku_base['Bonificado'].isna()]

                    df_m = df_sku_base.groupby(['FASEMANA', 'FASEMANA_DT'])[metrica].sum().reset_index().sort_values('FASEMANA_DT')

                    if not df_m.empty:
                        rango_completo = pd.date_range(start=df_m['FASEMANA_DT'].min(), end=df_m['FASEMANA_DT'].max(), freq='W-MON')
                        df_completo = pd.DataFrame({'FASEMANA_DT': rango_completo}); df_completo['FASEMANA'] = df_completo['FASEMANA_DT'].dt.strftime('%G-%V')
                        df_m = pd.merge(df_completo, df_m[['FASEMANA_DT', metrica]], on='FASEMANA_DT', how='left')
                        df_m[metrica] = df_m[metrica].fillna(0)

                    df_m.rename(columns={metrica: 'Valor_Real'}, inplace=True)
                    if df_m.empty: continue
                    df_m = df_m[['FASEMANA', 'FASEMANA_DT', 'Valor_Real']]

                    df_entrenamiento = aplicar_ajustes_atipicos(df_m, sku) if self.chk_aplicar_atipicos.isChecked() else df_m.copy()
                    serie = df_entrenamiento['Valor_Real'].values.astype(float); fechas_dt = df_entrenamiento['FASEMANA_DT']

                    sheet_name = str(sku)[:31].replace(':', '').replace('/', '').replace('\\', '').replace('?', '').replace('*', '').replace('[', '').replace(']', '')

                    if len(serie) <= semanas_eval + 4:
                        df_entrenamiento.to_excel(writer, sheet_name=sheet_name, index=False); continue

                    fechas_pred_futuro = pd.date_range(fechas_dt.iloc[-1] + pd.Timedelta(weeks=1), periods=semanas_futuro, freq='W-MON')
                    fechas_totales = list(fechas_dt) + list(fechas_pred_futuro)
                    fase_totales = list(df_entrenamiento['FASEMANA']) + [f.strftime('%G-%V') for f in fechas_pred_futuro]
                    real_totales = list(serie) + [np.nan] * semanas_futuro

                    df_out = pd.DataFrame({'Semana_ISO': fase_totales, 'Fecha': fechas_totales, 'Venta_Real': real_totales})

                    for mod_str in mods:
                        try:
                            pred = calcular_prediccion_modelo(serie, fechas_dt, fechas_pred_futuro, mod_str, semanas_eval, semanas_futuro)
                            if aplicar_promos:
                                pred_solo_futuro = pred[-semanas_futuro:]
                                pred_futura_ajustada = aplicar_promociones_futuras_array(pred_solo_futuro, fechas_pred_futuro, sku)
                                pred[-semanas_futuro:] = pred_futura_ajustada
                            pred_full = [np.nan] * (len(serie) - semanas_eval) + list(pred)
                            df_out[mod_str] = pred_full
                        except Exception as e: print(f"Error exportando modelo {mod_str} para SKU {sku}: {e}")

                    df_out.to_excel(writer, sheet_name=sheet_name, index=False); QApplication.processEvents()

            QMessageBox.information(self, "Éxito", "Exportación a Excel completada exitosamente.")

        except Exception as e: QMessageBox.critical(self, "Error de Exportación", f"Fallo al guardar:\n{str(e)}")
        finally: self.btn_exportar.setEnabled(True); self.btn_exportar.setText("📥 Exportar a Excel")

    # --- GENERAR ANÁLISIS ---
    def generar_grafico(self):
        skus = self.multi_sku.get_selected(); mods = self.multi_modelo.get_selected()
        if not skus or not mods: return
        self.btn_generar.setEnabled(False); QApplication.processEvents()

        self.t_error.setRowCount(0); self.t_pred.setRowCount(0); self.t_backtest.setRowCount(0)

        try:
            inicio = pd.to_datetime('2024-01-01') if self.date_desde.date().year() == 2022 else pd.to_datetime(self.date_desde.date().toPython())
            fin = pd.to_datetime(self.date_hasta.date().toPython())

            df_m = self.df_base[self.df_base['ITMCodigo'].astype(str).isin([str(s) for s in skus])]
            df_m = df_m[(df_m['FASEMANA_DT'].dt.date >= inicio.date()) & (df_m['FASEMANA_DT'].dt.date <= fin.date())]

            if df_m.empty:
                self.browser.setHtml("<h2 style='text-align: center; margin-top: 20%; color: #d62728;'>Sin datos.</h2>"); return

            if self.drop_bonificado.currentText() == "Excluir Bonificados" and 'Bonificado' in df_m.columns:
                bonificados = df_m['Bonificado'].astype(str).str.strip().str.lower()
                df_m = df_m[bonificados.isin(['0', 'no', 'false', 'nan', '<na>']) | df_m['Bonificado'].isna()]

            metrica = self.drop_unidad.currentText()
            df_m = df_m.groupby(['FASEMANA', 'FASEMANA_DT'])[metrica].sum().reset_index().sort_values('FASEMANA_DT')

            if not df_m.empty:
                rango_completo = pd.date_range(start=df_m['FASEMANA_DT'].min(), end=df_m['FASEMANA_DT'].max(), freq='W-MON')
                df_completo = pd.DataFrame({'FASEMANA_DT': rango_completo}); df_completo['FASEMANA'] = df_completo['FASEMANA_DT'].dt.strftime('%G-%V')
                df_m = pd.merge(df_completo, df_m[['FASEMANA_DT', metrica]], on='FASEMANA_DT', how='left')
                df_m[metrica] = df_m[metrica].fillna(0)

            df_m.rename(columns={metrica: 'Valor_Real'}, inplace=True)
            df_m = df_m[['FASEMANA', 'FASEMANA_DT', 'Valor_Real']]

            df_entrenamiento = aplicar_ajustes_atipicos(df_m, skus[0]) if self.chk_aplicar_atipicos.isChecked() else df_m.copy()

            html, errores, f_fut, f_eval, y_real = ejecutar_multiples_modelos_y_graficar(
                df_entrenamiento, skus[0], 'Valor_Real', mods, self.spin_futuro.value(), self.spin_eval.value(), self.chk_aplicar_promos.isChecked()
            )
            self.browser.setHtml(html)

            if not errores: return

            self.t_error.setRowCount(len(errores)); self.t_error.setColumnCount(4)
            self.t_error.setHorizontalHeaderLabels(["Modelo", "MAE", "WAPE", "Total"])
            for r, d in enumerate(errores):
                self.t_error.setItem(r,0,QTableWidgetItem(d['Modelo']))
                self.t_error.setItem(r,1,QTableWidgetItem(f"{d['MAE']:,.0f}"))
                self.t_error.setItem(r,2,QTableWidgetItem(f"{d['Error_Pct']:.1f}%"))
                self.t_error.setItem(r,3,QTableWidgetItem(f"{d['Total_Proyectado']:,.0f}"))

            mejores_modelos = errores[:2]
            if mejores_modelos and f_fut:
                self.t_pred.setColumnCount(1 + len(mejores_modelos))
                self.t_pred.setHorizontalHeaderLabels(["Semana"] + [m["Modelo"] for m in mejores_modelos])
                self.t_pred.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
                self.t_pred.setRowCount(len(f_fut))

                for i, fecha_str in enumerate(f_fut):
                    it_f = QTableWidgetItem(fecha_str); it_f.setTextAlignment(Qt.AlignCenter); self.t_pred.setItem(i, 0, it_f)
                    for j, modelo_data in enumerate(mejores_modelos):
                        it_val = QTableWidgetItem(f"{modelo_data['Pred_Futuro'][i]:,.1f}"); it_val.setTextAlignment(Qt.AlignCenter)
                        self.t_pred.setItem(i, j+1, it_val)

            if mejores_modelos and f_eval:
                self.t_backtest.setColumnCount(len(f_eval))
                self.t_backtest.setHorizontalHeaderLabels(f_eval)
                self.t_backtest.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeToContents)

                row_labels = ["Venta Real"]
                for m in mejores_modelos: row_labels.extend([f"➤ {m['Modelo']}", f"⚠️ {m['Modelo']} (Error)"])
                self.t_backtest.setRowCount(len(row_labels)); self.t_backtest.setVerticalHeaderLabels(row_labels)

                for col, val_real in enumerate(y_real):
                    it_real = QTableWidgetItem(f"{val_real:,.1f}"); it_real.setTextAlignment(Qt.AlignCenter); it_real.setBackground(QBrush(QColor("#e6f2ff")))
                    self.t_backtest.setItem(0, col, it_real)

                row_idx = 1
                for m in mejores_modelos:
                    for col, (val_real, val_pred) in enumerate(zip(y_real, m["Pred_Eval"])):
                        err_abs = abs(val_real - val_pred)
                        it_pred = QTableWidgetItem(f"{val_pred:,.1f}"); it_pred.setTextAlignment(Qt.AlignCenter)
                        it_err = QTableWidgetItem(f"{err_abs:,.1f}"); it_err.setTextAlignment(Qt.AlignCenter)
                        if val_real > 0:
                            if err_abs <= (val_real * 0.15): it_err.setForeground(QBrush(QColor("#2ca02c")))
                            elif err_abs > (val_real * 0.50): it_err.setForeground(QBrush(QColor("#d62728")))
                        self.t_backtest.setItem(row_idx, col, it_pred); self.t_backtest.setItem(row_idx + 1, col, it_err)
                    row_idx += 2

        except Exception as e:
            QMessageBox.critical(self, "Error", str(e))
        finally:
            self.btn_generar.setEnabled(True); self.btn_generar.setText("Generar Análisis")

if __name__ == "__main__":
    ruta = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Modelos de Prediccion\Resultado_Ventas_Agrupado_Semanal.csv"
    try:
        df = pd.read_csv(ruta, sep=';', encoding='utf-8-sig')
        df['FASEMANA_DT'] = pd.to_datetime(df['FASEMANA'] + '-1', format='%G-%V-%w', errors='coerce')
    except Exception as e: sys.exit(1)

    app = QApplication.instance()
    if not app: app = QApplication(sys.argv)
    ventana = DashboardApp(df); ventana.show()
    try: app.exec()
    except AttributeError: app.exec_()